# Benchmark: `buah_sayur` vs `fruit_veg`

Menguji **mana yang lebih bagus** secara nyata — akurasi, per-kelas, kecepatan, ukuran.

Dua kandidat:
- `model_kesegaran_buah_sayur.keras` — CNN polos, 150x150, 4.829.900 param, 55,32 MB
- `model_kesegaran_fruit_veg.keras` — MobileNetV2, 224x224, 2.273.356 param, 9,35 MB

> Cara upload: cell 3. Taruh kedua file `.keras` di satu folder di laptop,
> lalu pilih keduanya saat `files.upload()` muncul.

**Penting:** sel ini memakai **test split yang sama persis** untuk kedua model,
jadi perbandingannya adil.

In [ ]:
!pip install -q tensorflow==2.20.0 kagglehub scikit-learn matplotlib seaborn


In [ ]:
import tensorflow as tf, numpy as np, random, time, os, json
from pathlib import Path
import kagglehub

print("TF:", tf.__version__)
gpus = tf.config.list_physical_devices("GPU")
print("GPU:", tf.config.experimental.get_device_details(gpus[0])["name"] if gpus else "TIDAK ADA")

SEED = 123          # seed HARUS sama dgn notebook training
VAL_SPLIT = 0.2
random.seed(SEED); np.random.seed(SEED); tf.random.set_seed(SEED)


In [ ]:
from google.colab import files

print("Upload DUA file .keras sekaligus:")
up = files.upload()
names = [n for n in up if n.endswith(".keras")]
assert len(names) >= 2, f"Butuh 2 file .keras, dapat {len(names)}: {names}"

os.makedirs("/content/models", exist_ok=True)
for n, content in up.items():
    if n.endswith(".keras"):
        open(f"/content/models/{n}", "wb").write(content)
        print("tersimpan:", n)

MODELS = {}
for f_ in sorted(os.listdir("/content/models")):
    if f_.endswith(".keras"):
        MODELS[f_.replace("model_kesegaran_", "").replace(".keras", "")] = f"/content/models/{f_}"

print("\nModel yang diuji:")
for k, v in MODELS.items():
    print(f"  - {k}: {v}")


In [ ]:
HANDLE = "raghavrpotdar/fresh-and-stale-images-of-fruits-and-vegetables"
root = Path(kagglehub.dataset_download(HANDLE))

# dataset hasil download berupa folder kelas langsung (12 subfolder)
DATA = None
for p in [root] + [d for d in root.rglob("*") if d.is_dir()]:
    if p.is_dir() and len([d for d in p.iterdir() if d.is_dir()]) == 12:
        DATA = p; break

CLASSES = sorted([d.name for d in DATA.iterdir() if d.is_dir()])
print("DATA   :", DATA)
print("KELAS  :", len(CLASSES))
for i, c in enumerate(CLASSES):
    print(f"  {i:>2} {c:<24} {len(list((DATA/c).iterdir())):>5}")


In [ ]:
# TEST SET: split yang PERSIS SAMA dengan notebook training
# image_dataset_from_directory ikut menentukan seed saat split,
# jadi selama seed + validation_split sama, test set-nya identik.
_, test_ds = tf.keras.utils.image_dataset_from_directory(
    str(DATA),
    labels="inferred",
    label_mode="categorical",
    image_size=(224, 224),          # resize ke 224; model 150 tetap bisa (di-resize lagi)
    batch_size=64,
    validation_split=VAL_SPLIT,
    subset="validation",
    seed=SEED,
    shuffle=True,
    color_mode="rgb",
)

print("test classes:", test_ds.class_names)
assert test_ds.class_names == CLASSES
print("test batches:", tf.data.experimental.cardinality(test_ds).numpy())
print("class_names.json (urutan label):", CLASSES)


In [ ]:
# label benar disimpan terpisah supaya bisa dipakai backend
with open("class_names.json", "w") as f:
    json.dump({str(i): c for i, c in enumerate(CLASSES)}, f, indent=4)
print("class_names.json ditulis")


## Muat kedua model + cek apakah bisa di-inference

Catatan: `model_kesegaran_buah_sayur.keras` menyimpan optimizer state (66,7% file).
Kalau `load_model` gagal karena itu, muat dengan `compile=False`.

In [ ]:
loaded = {}
for name, path in MODELS.items():
    for compile_flag in (False, True):
        try:
            m = tf.keras.models.load_model(path, compile=compile_flag)
            loaded[name] = m
            print(f"{name:<28} OK (compile={compile_flag})  params={m.count_params():,}")
            break
        except Exception as e:
            print(f"{name:<28} GAGAL (compile={compile_flag}): {str(e)[:120]}")
            if not compile_flag:
                print("   -> coba ekstrak bobot manual")
    else:
        raise SystemExit(f"{name} tidak bisa dimuat")

print()
for name, m in loaded.items():
    print(f"{name:<28} input={m.input_shape}  output={m.output_shape}")


In [ ]:
# Kalau model 55 MB gagal dimuat, ambil trainable_weights-nya aja.
if len(loaded) < len(MODELS):
    for name, path in MODELS.items():
        if name in loaded:
            continue
        print(f"\n--- recovery {name} ---")
        m = tf.keras.models.load_model(path, compile=False)
        cfg = m.get_config()
        clean = tf.keras.Sequential.from_config(cfg)
        clean.build(m.input_shape)
        clean.set_weights(m.get_weights())
        loaded[name] = clean
        print("berhasil:", name, clean.count_params())


## Jalankan benchmark

Untuk setiap model:
1. **Akurasi** pada test set
2. **Laporan per-kelas** (lihat kelas mana yang lemah)
3. **Kecepatan inferensi** (waktu per gambar)
4. **Ukuran file**

In [ ]:
TEST_LABELS = np.concatenate([np.argmax(y.numpy(), axis=1) for _, y in test_ds])
print("total test:", len(TEST_LABELS))

results = {}
for name, m in loaded.items():
    t0 = time.time()
    probs = m.predict(test_ds, verbose=0)
    dt = time.time() - t0

    pred = probs.argmax(axis=1)
    acc = float((pred == TEST_LABELS).mean())

    results[name] = dict(
        acc=acc,
        pred=pred,
        probs=probs,
        ms_per_img=dt / len(TEST_LABELS) * 1000,
        params=int(m.count_params()),
        size_mb=os.path.getsize(MODELS[name]) / 1048576,
        input=m.input_shape[1],
    )
    print(f"{name:<28} acc={acc:.4f}  {dt:6.1f}s  {results[name]['ms_per_img']:.1f} ms/img  "
          f"{results[name]['params']:,} param  {results[name]['size_mb']:.2f} MB")


In [ ]:
from sklearn.metrics import classification_report

best = max(results, key=lambda k: results[k]["acc"])
print("=" * 78)
print(f"  PEMENANG: {best}  (akurasi {results[best]['acc']:.4f})")
print("=" * 78)
for name, r in results.items():
    star = " <-- MENANG" if name == best else ""
    print(f"  {name:<28} acc={r['acc']:.4f}  {r['ms_per_img']:6.1f} ms/img  "
          f"{r['size_mb']:6.2f} MB  {r['params']:>10,} param{star}")

print()
for name, r in results.items():
    print("=" * 78)
    print(f"LAPORAN PER-KELAS - {name}  (overall {r['acc']:.4f})")
    print("=" * 78)
    print(classification_report(TEST_LABELS, r['pred'], target_names=CLASSES, zero_division=0))


In [ ]:
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix

fig, axes = plt.subplots(1, len(results) + 1, figsize=(7 * (len(results) + 1), 7))
for ax, (name, r) in zip(axes, results.items()):
    cm = confusion_matrix(TEST_LABELS, r['pred'], labels=range(len(CLASSES)))
    im = ax.imshow(cm, cmap="Blues")
    ax.set_xticks(range(len(CLASSES))); ax.set_xticklabels(CLASSES, rotation=90, fontsize=7)
    ax.set_yticks(range(len(CLASSES))); ax.set_yticklabels(CLASSES, fontsize=7)
    for i in range(len(CLASSES)):
        for j in range(len(CLASSES)):
            ax.text(j, i, str(cm[i, j]), ha="center", va="center", fontsize=6)
    ax.set_title(f"{name}\nacc={r['acc']:.4f}")
    fig.colorbar(im, ax=ax, fraction=0.046)

ax = axes[-1]
ax.plot([r["ms_per_img"] for r in results.values()], "o")
ax.set_xticks(range(len(results)))
ax.set_xticklabels(list(results), rotation=20, ha="right", fontsize=8)
ax.set_ylabel("ms / gambar (lebih kecil lebih baik)")
ax.set_title("Kecepatan inferensi")
ax.grid(alpha=.3)
plt.tight_layout()
plt.show()


## Cek urutan label (penting!)

Dua model dilatih terpisah. Kalau urutan labelnya beda, backend akan salah
menampilkan nama kelas walau akurasinya tinggi.

Di bawah ini kedua model diuji pada 40 gambar yang **kelasnya sudah diketahui**
dari nama folder. Kalau urutan labelnya sama, akurasi kedua model di sini
hanya berbeda sedikit, karena ini data yang mereka pakai untuk training.

In [ ]:
# 40 gambar per kelas dari TRAIN, cek apakah prediksi = nama folder
sample_paths, sample_true = [], []
for i, c in enumerate(CLASSES):
    files = sorted(list((DATA / c).iterdir()))
    for fp in files[:10]:
        if fp.suffix.lower() in (".jpg", ".jpeg", ".png"):
            sample_paths.append(str(fp)); sample_true.append(i)

print("total sampel cek-label:", len(sample_paths))


def preprocess(paths, size):
    out = []
    for p in paths:
        img = tf.io.read_file(p)
        img = tf.image.decode_image(img, channels=3, expand_animations=False)
        img = tf.image.resize(img, (size, size), antialias=True)
        out.append(tf.cast(img, tf.float32) / 255.0)
    return np.stack(out)


y_true_lbl = np.array(sample_true)
label_report = {}
for name, m in loaded.items():
    size = m.input_shape[1]
    X = preprocess(sample_paths, size)
    p = m.predict(X, verbose=0)
    pred = p.argmax(axis=1)
    a = float((pred == y_true_lbl).mean())
    label_report[name] = (a, pred)
    print(f"{name:<28} akurasi cek-label = {a:.4f}")

print("\nCatatan: akurasi di sini TIDAK mengukur GENERALISASI, hanya memvalidasi")
print(" bahwa index prediksi -> index folder konsisten (label tidak tertukar).")
print(" Nilai tinggi (>0.8) = urutan label aman. Nilai rendah = perlu dicek manual.")


In [ ]:
for name, (a, pred) in label_report.items():
    print("=" * 70)
    print(f"{name}  -- akurasi cek-label {a:.4f}")
    print("=" * 70)
    wrong = [(CLASSES[t], CLASSES[p]) for t, p in zip(y_true_lbl, pred) if t != p]
    if not wrong:
        print("  SEMUA KECOCOKAN - urutan label konsisten")
    else:
        print(f"  {len(wrong)} salah dari {len(y_true_lbl)}")
        from collections import Counter
        for (t, p), n in Counter(wrong).most_common(12):
            print(f"    {t:<24} -> {p}")


## Ekspor model pemenang (bersih + label tertanam)

In [ ]:
# ===================== GANTI KE NAMA PEMENANG =====================
WINNER = best
# ==================================================================

m = loaded[WINNER]
m.class_names = CLASSES
m.save("model_kesegaran_best.keras", include_optimizer=True)   # ada optimizer state
m.save("model_kesegaran_best.h5", overwrite=True)              # format lama, kompatibel

# Versi ringan: tanpa optimizer, jauh lebih kecil
lean = tf.keras.models.clone_model(m)
lean.set_weights(m.get_weights())
lean.class_names = CLASSES
lean.save("model_kesegaran_best_lean.keras", include_optimizer=False)

for f_ in ["model_kesegaran_best.keras", "model_kesegaran_best_lean.keras",
           "model_kesegaran_best.h5", "class_names.json"]:
    print(f"{f_:<40} {Path(f_).stat().st_size/1048576:>7.2f} MB")

print("\nHASIL AKHIR")
print(f"  pemenang : {WINNER}")
print(f"  akurasi  : {results[WINNER]['acc']:.4f}")


In [ ]:
from google.colab import files
for fn in ["model_kesegaran_best.keras", "model_kesegaran_best_lean.keras",
           "model_kesegaran_best.h5", "class_names.json"]:
    try:
        files.download(fn)
    except Exception as e:
        print(fn, "gagal:", e)
print("Selesai.")
